# ESMFold2 — Antibody-Antigen Complex Prediction

Predict the 3D structure of an antibody-antigen complex using ESMFold2-Fast via the Biohub Forge API. No GPU required — computation runs on Biohub's servers.

In [ ]:
# Install the ESM package
!pip install "esm@git+https://github.com/mahdishafiei/esmfold2_relaxed.git@main" -q

from esm.sdk.forge import SequenceStructureForgeInferenceClient
from esm.utils.structure.input_builder import ProteinInput, StructurePredictionInput
from esm.sdk.api import FoldingConfig, ESMProteinError
from esm.utils.constants.models import ESMFOLD2_FAST
import os, concurrent.futures

## Step 1 — Enter your API token

In [ ]:
from getpass import getpass
BIOHUB_TOKEN = getpass("Enter your Biohub API token: ")

## Step 2 — Enter your sequences

Provide heavy chain (VH), light chain (VL), and antigen sequences. Any chain is optional.

In [ ]:
HEAVY   = "EVQLVESGGGLVKPGGSLRLSCAASGFTFSSYSMNWVRQAPGKGLEWVSSISSSSSYIYYADSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCARAPAAISYYMDVWGKGTTVTVSG"
LIGHT   = "DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKPGQPPKLLIYWASTRESGVPDRFSGSGSGTDFTLTISSLQAEDVAVYYCQQYYSTPALTFGGGTKVEIK"
ANTIGEN = "DQICIGYHANNSTEQVDTIMEKNVTVTHAQDILEKKHNGKLCDLDGVKPLILRDCSVAGWLLGNPMCDEFINVPEWSYIVEKANPVNDLCFPGDFNDYEELKHLLSRINHFEKIQIIPKSSWSSHEASLGVSSACPYQGKSSFFRNVVWLIKKNSTYPTIKRSYNNTNQEDLLVLWGIHHPNDAAEQTKLYQNPTTYISVGTSTLNQRLVPRIATRSKVNGQSGRMEFFWTILKPNDAINFESNGNFIAPEYAYKIVKKGDSTIMKSELEYGNCNTKCQTPMGAINSSMPFHNIHPLTIGECPKYVKSNRLVLATGLRNSPQRERGLFGAIAGFIEGGWQGMVDGWYGYHHSNEQGSGYAADKESTQKAIDGVTNKVNSIIDKMNTQFEAVGREFNNLERRIENLNKKMEDGFLDVWTYNAELLVLMENERTLDFHDSNVKNLYDKVRLQLRDNAKELGNGCFEFYHKCDNECMESVRNGTYDYPQYSEEARLKREEISGVR"
NUM_SEEDS = 5  # increase for better results (costs more credits)

provided = {k: v.strip() for k, v in [("heavy", HEAVY), ("light", LIGHT), ("antigen", ANTIGEN)] if v.strip()}
total_aa = sum(len(s) for s in provided.values())
print(f"Chains: {', '.join(f'{k}={len(v)}aa' for k,v in provided.items())}")
print(f"Total residues: {total_aa} (limit: 768)")
assert total_aa <= 768, f"Total length {total_aa} exceeds 768 residue limit. Trim your sequences."

## Step 3 — Run prediction

In [ ]:
def run_seed(seed_idx):
    client = SequenceStructureForgeInferenceClient(token=BIOHUB_TOKEN, model=ESMFOLD2_FAST)
    config = FoldingConfig(num_loops=20, num_sampling_steps=100, include_pae=True)
    inputs = StructurePredictionInput(
        sequences=[ProteinInput(id=name, sequence=seq) for name, seq in provided.items()]
    )
    result = client.fold_all_atom(inputs, config=config)
    if isinstance(result, ESMProteinError):
        print(f"  seed {seed_idx}: ERROR — {result}")
        return None
    iptm = result.iptm or 0.0
    ptm  = result.ptm  or 0.0
    print(f"  seed {seed_idx}: ipTM={iptm:.3f}  pTM={ptm:.3f}")
    return result

print(f"Running {NUM_SEEDS} seeds (sequential to respect rate limits)...")
results = []
for i in range(1, NUM_SEEDS + 1):
    r = run_seed(i)
    if r is not None:
        results.append(r)

if not results:
    raise RuntimeError("All seeds failed.")

best = max(results, key=lambda r: r.iptm or 0.0)
print(f"\nBest: ipTM={best.iptm:.3f}  pTM={best.ptm:.3f}  mean pLDDT={float(best.plddt.mean()):.3f}")

## Step 4 — Save and download

In [ ]:
output_file = "complex.cif"
with open(output_file, "w") as f:
    f.write(best.complex.to_mmcif())
print(f"Saved → {output_file}")

# Download in Colab
from google.colab import files
files.download(output_file)

## Score interpretation

| ipTM | Meaning |
|---|---|
| > 0.8 | Confident — trust the pose |
| 0.5–0.8 | Plausible — inspect carefully |
| < 0.5 | Low confidence — try more seeds |

**Note:** Open the `.cif` file in PyMOL or ChimeraX.